# Experiment 07 — Reset ratchet: memory accumulation under periodic plaque removal

The periodontal cue cannot be eliminated. Plaque regrows on a non-shedding surface and is removed only periodically, by home care or professional debridement. This experiment asks what response-state memory does under that forcing regime.

**Model.** M1 is written in memory coordinates $A=\ln(\theta_0/\theta)$:

$$\dot H=-H+F(M+u(t);\,\theta_0e^{-A}),\qquad \dot M=r\,(-M+G(H)),\qquad \dot A=\varepsilon\,(\alpha H-A)$$

This is algebraically identical to $\dot\theta=\varepsilon\theta(\ln(\theta_0/\theta)-\alpha H)$.

**Forcing.** Plaque regrows toward a ceiling and is reset every $T$:
$u(t)=u_{\max}(1-e^{-k\tau})$, with $\tau=t \bmod T$.
Results are reported as the **peak plaque reached before each reset**, $u_{\text{peak}}=u_{\max}(1-e^{-kT})$, so that a static analysis predicts conversion when $u_{\text{peak}}$ exceeds the upward fold $u_{\uparrow}=0.4682$.

**Hosts compared** (comparator backbone of Exp 06B: $n=q=4$, $K_M=0.9$, $\theta_0=0.8$, $r=1$, $\alpha=1$):
* **slow memory**, $\varepsilon \in \{0.3, 0.1, 0.03, 0.01, 0.003\}$;
* **instant adaptation**, $A=\alpha H$ at every instant ($\varepsilon\to\infty$). This host has the **same equilibria and the same bifurcation diagram** as M1 but no memory timescale;
* **frozen memory** ($\varepsilon=0$, $A$ held fixed). This is used only for the adiabatic reduction.

**Entrapment.** A cycle is *entrapped* if $H$ never falls below 0.5 during it, meaning the reset no longer resolves inflammation. *Conversion* is the first cycle from which every later cycle is entrapped.

Sections:
* **A** example ratchets
* **B** adiabatic stroboscopic map and the $\varepsilon\to 0$ threshold
* **C** phase maps over (T, peak plaque, ε)
* **D** latency scaling near threshold
* **E** debridement variant (M also reset)
* **F** consistency checks and summary

All numerical outputs and figures are saved to `Biosemiotic_Perio/Experiment_07_Reset_Ratchet`.

In [ ]:
# =====================================================================
# EXPERIMENT 07 — folder setup
# =====================================================================
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/Biosemiotic_Perio")
except ImportError:                      # local execution
    BASE_DIR = Path.cwd() / "Biosemiotic_Perio"

EXP_DIR = BASE_DIR / "Experiment_07_Reset_Ratchet"
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Experiment folder:\n{EXP_DIR}\nExists: {EXP_DIR.exists()}")

In [ ]:
import sys, json, time, platform, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
import scipy
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit

versions_text = f"""
Experiment: 07 Reset ratchet (periodic plaque regrowth and removal)
Created: {datetime.now(timezone.utc).isoformat()}

Python: {sys.version}
Platform: {platform.platform()}
NumPy: {np.__version__}
Pandas: {pd.__version__}
SciPy: {scipy.__version__}
Matplotlib: {matplotlib.__version__}
"""
with open(EXP_DIR / "software_versions.txt", "w") as f:
    f.write(versions_text)
print(versions_text)

In [ ]:
# =====================================================================
# PARAMETERS (dimensionless, exploratory — not empirical estimates)
# =====================================================================
PARAMS = dict(n=4.0, q=4.0, K=0.9, th0=0.8, r=1.0, a=1.0)   # Exp 06B comparator backbone
K_REGROW = 0.05          # plaque regrowth rate k  (regrowth time 1/k = 20)
DT = 0.1                 # RK4 step; every T is a multiple of DT
ENTRAP_H = 0.5           # entrapment criterion on the cycle minimum of H
U_UP = 0.4682392215      # static upward fold of the comparator (Exp 06B)
U_DOWN = 0.0630204768    # static downward fold

T_LIST = np.array([10, 15, 20, 30, 40, 60, 80, 120, 160], float)   # reset intervals
EPS_LIST = [0.3, 0.1, 0.03, 0.01, 0.003]                          # memory rates
PEAK = np.round(np.arange(0.30, 0.9001, 0.01), 4)                 # peak plaque grid

with open(EXP_DIR / "parameters.json", "w") as f:
    json.dump(dict(backbone=PARAMS, k_regrow=K_REGROW, dt=DT, entrap_H=ENTRAP_H,
                   u_up=U_UP, u_down=U_DOWN, T_list=T_LIST.tolist(), eps_list=EPS_LIST,
                   peak_grid=[float(PEAK[0]), float(PEAK[-1]), 0.01]), f, indent=4)

# ---- figure style -----------------------------------------------------
INK, INK2, GRIDC, SURF, NONE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#f0efec"
MEM_COL = {0.3: "#86b6ef", 0.1: "#5598e7", 0.03: "#2a78d6", 0.01: "#1c5cab", 0.003: "#0d366b"}
INSTANT_COL = "#eb6834"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#3987e5",
                                                    "#1c5cab", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.6,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})
def save_fig(fig, name):
    fig.savefig(EXP_DIR / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================================
# CORE SIMULATOR — vectorised fixed-step RK4 over parameter grids
# =====================================================================
def hill(x, h, n):
    x = np.maximum(x, 0.0)
    xn = x ** n
    return xn / (xn + h ** n)


def _rhs(H, M, A, u, eps, p, mode):
    A_eff = p['a'] * H if mode == 'instant' else A
    th = p['th0'] * np.exp(-A_eff)
    dH = -H + hill(M + u, th, p['n'])
    dM = p['r'] * (-M + hill(H, p['K'], p['q']))
    dA = eps * (p['a'] * H - A) if mode == 'adaptive' else np.zeros_like(A)
    return dH, dM, dA


def simulate(umax, T, eps=0.0, horizon=None, n_cycles=None, mode='adaptive',
             reset_M=False, y0=None, p=PARAMS, k=K_REGROW, dt=DT):
    """
    Integrate many (umax, T, eps, initial state) combinations in parallel.
    mode: 'adaptive' (M1), 'instant' (A = alpha*H), 'frozen' (A held at y0).
    reset_M: also set M = 0 at every reset (professional debridement).
    Returns per-cycle arrays (..., n_cycles): theta_reset, Hmin, Hmax, Hmean,
    frac_high (fraction of cycle with H > ENTRAP_H), relapse_time (time after
    the reset at which H rises through ENTRAP_H; NaN if it does not).
    """
    if y0 is None:
        y0 = (0.0, 0.0, 0.0)
    H0, M0, A0, umax, T, eps = np.broadcast_arrays(*[np.atleast_1d(np.asarray(v, float))
                                                    for v in (*y0, umax, T, eps)])
    shape = umax.shape
    umax, T, eps = umax.ravel().copy(), T.ravel().copy(), eps.ravel().copy()
    H, M, A = H0.ravel().copy(), M0.ravel().copy(), A0.ravel().copy()
    n_el = umax.size
    Ns = np.rint(T / dt).astype(np.int64)
    if np.any(np.abs(Ns * dt - T) > 1e-9):
        raise ValueError("T must be a multiple of dt")
    if horizon is None:
        horizon = n_cycles * T.max()
    n_steps = int(np.rint(horizon / dt))
    n_cyc = int(n_steps // Ns.min()) + 1
    keys = ('theta_reset', 'Hmin', 'Hmax', 'Hmean', 'frac_high', 'relapse_time')
    rec = {key: np.full((n_el, n_cyc), np.nan) for key in keys}
    Hmin, Hmax = H.copy(), H.copy()
    Hint, Thigh = np.zeros(n_el), np.zeros(n_el)
    t_rel = np.full(n_el, np.nan)
    idx = np.arange(n_el)
    u_at = lambda tau: umax * (1.0 - np.exp(-k * tau))

    for i in range(n_steps):
        j = i % Ns
        c = i // Ns
        start = (j == 0)
        if start.any():
            if reset_M and i > 0:
                M = np.where(start, 0.0, M)
            A_now = p['a'] * H if mode == 'instant' else A
            rec['theta_reset'][idx[start], c[start]] = p['th0'] * np.exp(-A_now[start])
            Hmin = np.where(start, H, Hmin); Hmax = np.where(start, H, Hmax)
            Hint = np.where(start, 0.0, Hint); Thigh = np.where(start, 0.0, Thigh)
            t_rel = np.where(start, np.nan, t_rel)
        tau0 = j * dt
        u0, uh, u1 = u_at(tau0), u_at(tau0 + 0.5 * dt), u_at(tau0 + dt)
        k1 = _rhs(H, M, A, u0, eps, p, mode)
        k2 = _rhs(H + .5*dt*k1[0], M + .5*dt*k1[1], A + .5*dt*k1[2], uh, eps, p, mode)
        k3 = _rhs(H + .5*dt*k2[0], M + .5*dt*k2[1], A + .5*dt*k2[2], uh, eps, p, mode)
        k4 = _rhs(H + dt*k3[0], M + dt*k3[1], A + dt*k3[2], u1, eps, p, mode)
        Hn = H + dt/6 * (k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
        M = M + dt/6 * (k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
        A = A + dt/6 * (k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
        Hint += 0.5 * (H + Hn) * dt
        Thigh += dt * (Hn > ENTRAP_H)
        newly = np.isnan(t_rel) & (H <= ENTRAP_H) & (Hn > ENTRAP_H)
        t_rel = np.where(newly, (j + 1) * dt, t_rel)
        H = Hn
        Hmin = np.minimum(Hmin, H); Hmax = np.maximum(Hmax, H)
        end = (j == Ns - 1)
        if end.any():
            r_, c_ = idx[end], c[end]
            rec['Hmin'][r_, c_] = Hmin[end]; rec['Hmax'][r_, c_] = Hmax[end]
            rec['Hmean'][r_, c_] = Hint[end] / T[end]
            rec['frac_high'][r_, c_] = Thigh[end] / T[end]
            rec['relapse_time'][r_, c_] = t_rel[end]
    out = {key: v.reshape(*shape, n_cyc) for key, v in rec.items()}
    if mode == 'instant':
        A = p['a'] * H
    out['final'] = tuple(v.reshape(shape) for v in (H, M, A))
    return out


def conversion_cycle(Hmin_rec, thr=ENTRAP_H):
    """First cycle from which every later completed cycle is entrapped; NaN if none."""
    shp = Hmin_rec.shape[:-1]
    Hm = Hmin_rec.reshape(-1, Hmin_rec.shape[-1])
    res = np.full(Hm.shape[0], np.nan)
    for e in range(Hm.shape[0]):
        done = np.where(~np.isnan(Hm[e]))[0]
        if done.size == 0:
            continue
        row = Hm[e, :done[-1] + 1] > thr
        if not row[-1]:
            continue
        non = np.where(~row)[0]
        res[e] = 0 if non.size == 0 else non[-1] + 1
    return res.reshape(shp)


def grid_threshold(conv, grid):
    """Smallest grid value with conversion, per row; NaN if none."""
    return np.array([grid[np.argmax(~np.isnan(r))] if (~np.isnan(r)).any() else np.nan
                     for r in conv])

peak_to_umax = lambda peak, T: peak / (1 - np.exp(-K_REGROW * T))

In [ ]:
# =====================================================================
# VALIDATION — vectorised RK4 vs LSODA (theta form, integrated cycle by cycle)
# =====================================================================
def rhs_theta(t, y, um, eps, p=PARAMS):
    H, M, th = y
    u = um * (1 - np.exp(-K_REGROW * t))
    return [-H + hill(M + u, th, p['n']), p['r'] * (-M + hill(H, p['K'], p['q'])),
            eps * th * (np.log(p['th0'] / th) - p['a'] * H)]

cases = [(0.575, 60.0, 0.01), (0.56, 60.0, 0.01), (0.75, 20.0, 0.1), (0.62, 40.0, 0.003)]
NC = 30
sim = simulate([peak_to_umax(c[0], c[1]) for c in cases], [c[1] for c in cases],
               [c[2] for c in cases], horizon=NC * 60.0)
rows = []
for e, (pk, T, eps) in enumerate(cases):
    y = np.array([0.0, 0.0, PARAMS['th0']]); th_ref, hmin_ref = [], []
    for c in range(int(NC * 60 / T)):
        th_ref.append(y[2])
        sol = solve_ivp(rhs_theta, (0, T), y, args=(peak_to_umax(pk, T), eps), method='LSODA',
                        rtol=1e-10, atol=1e-12, dense_output=True)
        hmin_ref.append(sol.sol(np.linspace(0, T, 2001))[0].min()); y = sol.y[:, -1]
    n = len(th_ref)
    rows.append(dict(peak=pk, T=T, eps=eps, cycles=n,
                     max_abs_theta_diff=np.nanmax(np.abs(sim['theta_reset'][e, :n] - th_ref)),
                     max_abs_Hmin_diff=np.nanmax(np.abs(sim['Hmin'][e, :n] - hmin_ref))))
val_df = pd.DataFrame(rows)
val_df.to_csv(EXP_DIR / "validation_vs_LSODA.csv", index=False)
display(val_df)

## A. Example ratchets

Reset interval $T=60$ ($kT=3$). For the slow-memory host ($\varepsilon=0.01$) the averaged threshold is a peak of about 0.570. The three peak-plaque levels sit just below it, just above it, and well above it. The instant host is run at the same levels.

In [ ]:
TA, EPS_A, NCA = 60.0, 0.01, 60
PEAKS_A = [0.56, 0.575, 0.62]
sA = simulate(peak_to_umax(np.array(PEAKS_A), TA), TA, EPS_A, n_cycles=NCA)
sAi = simulate(peak_to_umax(np.array(PEAKS_A), TA), TA, 0.0, n_cycles=NCA, mode='instant')
rows = []
for host, s in [('slow (eps=0.01)', sA), ('instant', sAi)]:
    for i, pk in enumerate(PEAKS_A):
        for c in range(NCA):
            rows.append(dict(host=host, peak=pk, cycle=c, theta_at_reset=s['theta_reset'][i, c],
                             H_peak=s['Hmax'][i, c], H_min=s['Hmin'][i, c], H_mean=s['Hmean'][i, c]))
dfA = pd.DataFrame(rows); dfA.to_csv(EXP_DIR / "fig07_01_example_ratchets_data.csv", index=False)
print("cycles to conversion  slow:", conversion_cycle(sA['Hmin']), "  instant:", conversion_cycle(sAi['Hmin']))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
lstyle = {0.56: ':', 0.575: '-', 0.62: '--'}
for i, pk in enumerate(PEAKS_A):
    cyc = np.arange(NCA)
    ax[0].plot(cyc, sA['theta_reset'][i, :NCA], lstyle[pk], color=MEM_COL[0.01], label=f"slow, peak {pk}")
    ax[1].plot(cyc, sA['Hmax'][i, :NCA], lstyle[pk], color=MEM_COL[0.01], label=f"slow, peak {pk}")
    ax[0].plot(cyc, sAi['theta_reset'][i, :NCA], lstyle[pk], color=INSTANT_COL, lw=1.1, label=f"instant, peak {pk}")
    ax[1].plot(cyc, sAi['Hmax'][i, :NCA], lstyle[pk], color=INSTANT_COL, lw=1.1, label=f"instant, peak {pk}")
ax[0].set(xlabel="maintenance cycle", ylabel=r"threshold $\theta$ at each reset",
          title="Memory carried across resets")
ax[1].set(xlabel="maintenance cycle", ylabel="peak host response in the cycle",
          title="Same plaque every cycle, rising response")
ax[1].legend(fontsize=7, ncol=2, loc="lower right")
save_fig(fig, "fig07_01_example_ratchets")

## B. Adiabatic stroboscopic map

Because $\dot A=\varepsilon(\alpha H-A)$ is linear in $A$, the value of $A$ at successive resets obeys **exactly**

$$A_{n+1}=\lambda A_n+(1-\lambda)\,\alpha\,\Theta_n,\qquad \lambda=e^{-\varepsilon T},$$

where $\Theta_n$ is an exponentially weighted average of $H$ over cycle $n$. When memory is slow ($\varepsilon \ll 1$), $H$ relaxes within each cycle to the periodic response of the frozen-memory system. $\Theta_n$ then becomes a function $\Theta(A_n)$, the cycle-mean of $H$ at frozen $A$, and the map is one-dimensional. Its fixed points solve $A^*=\alpha\Theta(A^*)$ and do not depend on $\varepsilon$; $\varepsilon$ sets only how fast they are approached.

Because $\Theta'\ge 0$ (monotone system), the multiplier $\lambda+(1-\lambda)\alpha\Theta'$ is positive. Stability can therefore be lost only through a saddle-node (tangency), never through period doubling.

A naive site starts at $A=0$. It stays healthy if $\alpha\Theta(A)$ meets the diagonal before the frozen low response ceases to exist at $A_c$. Otherwise memory accumulates until $A_c$ and the site converts.

In [ ]:
A_GRID = np.round(np.arange(0.0, 1.2001, 0.01), 4)
PEAK_TH = np.round(np.arange(0.30, 0.9501, 0.005), 4)
Theta = np.full((len(T_LIST), len(PEAK_TH), len(A_GRID)), np.nan)
low = np.zeros_like(Theta, bool)
t0 = time.time()
for iT, T in enumerate(T_LIST):
    PP, AA = np.meshgrid(PEAK_TH, A_GRID, indexing='ij')
    nc = int(max(10, np.ceil(400 / T)))
    s = simulate(peak_to_umax(PP, T), T, 0.0, n_cycles=nc, mode='frozen', y0=(0.0, 0.0, AA))
    Theta[iT] = s['Hmean'][..., nc - 1]
    low[iT] = s['Hmin'][..., nc - 1] <= ENTRAP_H
    print(f"T={T:5.0f}  ({time.time() - t0:5.0f}s)")

g = PARAMS['a'] * Theta - A_GRID
A_star = np.full(Theta.shape[:2], np.nan); A_c = np.full(Theta.shape[:2], np.nan)
for iT in range(len(T_LIST)):
    for ip in range(len(PEAK_TH)):
        lo = low[iT, ip]
        end = int(np.argmax(~lo)) if (~lo).any() else len(A_GRID)
        if end < len(A_GRID):
            A_c[iT, ip] = A_GRID[end]
        cross = np.where(g[iT, ip, :end] <= 0)[0]
        if cross.size and cross[0] > 0:
            i = cross[0]; g0, g1 = g[iT, ip, i - 1], g[iT, ip, i]
            A_star[iT, ip] = A_GRID[i - 1] + 0.01 * g0 / (g0 - g1)
peak_avg = grid_threshold(np.where(np.isnan(A_star), 1.0, np.nan), PEAK_TH)   # first peak with no slow equilibrium
theory_df = pd.DataFrame(dict(T=T_LIST, kT=K_REGROW * T_LIST, peak_threshold_eps0=peak_avg,
                              peak_threshold_over_static_fold=peak_avg / U_UP))
theory_df.to_csv(EXP_DIR / "averaged_map_thresholds.csv", index=False)
np.savez_compressed(EXP_DIR / "frozen_memory_cycle_response.npz", T_LIST=T_LIST, PEAK=PEAK_TH,
                    A_GRID=A_GRID, Theta=Theta, low=low, A_star=A_star, A_c=A_c)
display(theory_df)

In [ ]:
iT = list(T_LIST).index(60.0)
PEAKS_B = [0.55, 0.565, 0.58]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
rows = []
for pk, col in zip(PEAKS_B, ["#86b6ef", "#2a78d6", "#0d366b"]):
    ip = int(np.argmin(np.abs(PEAK_TH - pk)))
    m = low[iT, ip]
    ax[0].plot(A_GRID[m], PARAMS['a'] * Theta[iT, ip, m], color=col, label=f"peak {PEAK_TH[ip]:.3f}")
    if (~m).any():
        Ae = A_GRID[m][-1]; ax[0].plot(Ae, PARAMS['a'] * Theta[iT, ip, m][-1], 'x', color=col, ms=7, mew=1.6)
    for a_, th_, l_ in zip(A_GRID, Theta[iT, ip], m):
        rows.append(dict(T=60, peak=PEAK_TH[ip], A=a_, Theta=th_, low_response=l_))
ax[0].plot([0, 0.6], [0, 0.6], color=INK2, lw=1, ls='--', label=r"diagonal $A=\alpha\Theta$")
ax[0].set(xlim=(0, 0.6), ylim=(0, 0.6), xlabel=r"memory at reset $A=\ln(\theta_0/\theta)$",
          ylabel=r"$\alpha\,\Theta(A)$  (cycle-mean response)",
          title="Reduced map, T = 60 (x: low response ends)")
ax[0].legend(fontsize=7, loc="upper left")
pd.DataFrame(rows).to_csv(EXP_DIR / "fig07_02_reduced_map_data.csv", index=False)

# thresholds vs interval: static fold, instant host, averaged theory (filled in Section C)
ax[1].axhline(1.0, color=INK2, ls='--', lw=1, label="static fold (equilibrium analysis)")
ax[1].plot(K_REGROW * T_LIST, peak_avg / U_UP, color=INK, lw=1.8, label=r"slow memory, $\varepsilon\to0$ (theory)")
ax[1].set(xscale='log', xlabel="reset interval  kT  (units of plaque regrowth time)",
          ylabel="conversion threshold\n(peak plaque / static fold)", title="Where conversion begins")
FIG02 = (fig, ax)

## C. Phase maps over reset interval, peak plaque and memory timescale

Every host starts as a naive site, $(H,M,A)=(0,0,0)$. The horizon is $\max(30\,T_{\max},\,40/\varepsilon)$. Colour shows the number of cycles before conversion, and grey means no conversion within the horizon.

In [ ]:
TT, PP = np.meshgrid(T_LIST, PEAK, indexing='ij')
UMAX = peak_to_umax(PP, TT)
maps = {}
for eps in [np.inf] + EPS_LIST:
    t0 = time.time()
    if np.isinf(eps):
        s = simulate(UMAX, TT, 0.0, horizon=60 * T_LIST.max(), mode='instant'); key = 'instant'
    else:
        s = simulate(UMAX, TT, eps, horizon=max(30 * T_LIST.max(), 40 / eps)); key = eps
    maps[key] = conversion_cycle(s['Hmin'])
    print(f"{str(key):8s} done ({time.time() - t0:.0f}s)")

rows = []
for key, conv in maps.items():
    for iT, T in enumerate(T_LIST):
        for ip, pk in enumerate(PEAK):
            rows.append(dict(host='instant' if key == 'instant' else f'eps={key}', T=T, kT=K_REGROW*T,
                             peak=pk, peak_over_static_fold=pk / U_UP, cycles_to_conversion=conv[iT, ip]))
dfC = pd.DataFrame(rows); dfC.to_csv(EXP_DIR / "fig07_03_phase_maps_data.csv", index=False)

thr = {key: grid_threshold(conv, PEAK) for key, conv in maps.items()}
thr_df = pd.DataFrame({'T': T_LIST, 'kT': K_REGROW * T_LIST, 'static_fold': U_UP,
                       'instant': thr['instant'], **{f'eps={e}': thr[e] for e in EPS_LIST},
                       'eps->0 theory': peak_avg})
thr_df.to_csv(EXP_DIR / "conversion_thresholds_peak_plaque.csv", index=False)
display(thr_df.round(3))

In [ ]:
keys = ['instant'] + EPS_LIST
fig, axs = plt.subplots(2, 3, figsize=(10.5, 6.4), constrained_layout=True, sharex=True, sharey=True)
xi = np.arange(len(T_LIST)); norm = LogNorm(1, 400)
for axx, key in zip(axs.ravel(), keys):
    conv = maps[key]
    axx.pcolormesh(np.r_[xi - .5, xi[-1] + .5], np.r_[PEAK - .005, PEAK[-1] + .005] / U_UP,
                   np.where(np.isnan(conv), 1, np.nan).T, cmap=LinearSegmentedColormap.from_list('n', [NONE, NONE]),
                   shading='flat')
    pcm = axx.pcolormesh(np.r_[xi - .5, xi[-1] + .5], np.r_[PEAK - .005, PEAK[-1] + .005] / U_UP,
                         np.maximum(conv, 1).T, cmap=BLUES, norm=norm, shading='flat')
    axx.axhline(1.0, color=INK2, ls='--', lw=1)
    axx.plot(xi, peak_avg / U_UP, color=INK, lw=1.4)
    axx.plot(xi, thr['instant'] / U_UP, color=INSTANT_COL, lw=1.4)
    axx.set_title("instant adaptation" if key == 'instant' else rf"slow memory, $\varepsilon$ = {key}")
    axx.grid(False)
for axx in axs[-1]:
    axx.set_xticks(xi, [f"{v:g}" for v in K_REGROW * T_LIST]); axx.set_xlabel("reset interval kT")
for axx in axs[:, 0]:
    axx.set_ylabel("peak plaque / static fold")
cb = fig.colorbar(pcm, ax=axs, shrink=0.7, label="cycles before conversion (grey: none)")
axs[0, 0].text(0.02, 0.97, "black: slow-memory threshold (theory)\norange: instant-host threshold\n"
               "dashed: static fold", transform=axs[0, 0].transAxes, va='top', fontsize=6.5, color=INK2)
save_fig(fig, "fig07_03_phase_maps")

# complete Figure 02 right panel with simulated thresholds
fig2, ax2 = FIG02
ax2[1].plot(K_REGROW * T_LIST, thr['instant'] / U_UP, color=INSTANT_COL, lw=1.8,
            label="instant adaptation (same equilibria)")
for e in EPS_LIST:
    ax2[1].plot(K_REGROW * T_LIST, thr[e] / U_UP, 'o', ms=3.5, color=MEM_COL[e], label=rf"simulated, $\varepsilon$={e}")
ax2[1].legend(fontsize=6.5, loc="upper right")
thr_df.to_csv(EXP_DIR / "fig07_02_thresholds_data.csv", index=False)
save_fig(fig2, "fig07_02_reduced_map_and_thresholds")

## D. Latency near threshold

Near a saddle-node of the averaged map, the passage time diverges as the inverse square root of the distance from threshold and scales with the memory time $1/\varepsilon$. In cycles, the prediction is $N \approx c\,(\varepsilon T)^{-1}(u_{\text{peak}}-u_c)^{-1/2}$. Here $T=60$. Each threshold is located by grid refinement and then fitted together with the exponent.

In [ ]:
TD, NCYC_D = 60.0, 400
lat_rows, fit_rows = [], []
for key in ['instant', 0.03, 0.01, 0.003]:
    mode, eps = ('instant', 0.0) if key == 'instant' else ('adaptive', key)
    t0 = time.time(); lo, hi = 0.45, 0.65
    for _ in range(2):
        pk = np.linspace(lo, hi, 41)
        nc = conversion_cycle(simulate(peak_to_umax(pk, TD), TD, eps, n_cycles=NCYC_D, mode=mode)['Hmin'])
        i = int(np.argmax(~np.isnan(nc))); lo, hi = pk[i - 1], pk[i]
    dl = np.logspace(-4, -0.8, 20)
    nc = conversion_cycle(simulate(peak_to_umax(hi + dl, TD), TD, eps, n_cycles=NCYC_D, mode=mode)['Hmin'])
    for d_, n_ in zip(dl, nc):
        lat_rows.append(dict(host=str(key), T=TD, threshold_bracket_hi=hi, peak=hi + d_, cycles=n_))
    fit = dict(host=str(key), threshold_bracket=f"({lo:.5f}, {hi:.5f}]")
    if key != 'instant':
        f4 = lambda d, pc, a, b, gam: a + b * np.maximum(hi + d - pc, 1e-9) ** (-gam)
        popt, _ = curve_fit(f4, dl, nc, p0=(lo, 1.0, 1.0, 0.5),
                            bounds=([lo - 0.01, -50, 0, 0.1], [hi, 50, 1e3, 1.5]), maxfev=50000)
        fit.update(pc=popt[0], a=popt[1], b=popt[2], exponent=popt[3], b_times_epsT=popt[2] * eps * TD)
    fit_rows.append(fit)
    print(f"{str(key):8s} ({time.time() - t0:.0f}s)  {fit}")
dfD = pd.DataFrame(lat_rows); dfD.to_csv(EXP_DIR / "fig07_04_latency_scaling_data.csv", index=False)
fitD = pd.DataFrame(fit_rows); fitD.to_csv(EXP_DIR / "latency_scaling_fits.csv", index=False)
display(fitD)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
for key in ['instant', 0.03, 0.01, 0.003]:
    d = dfD[dfD.host == str(key)]
    col = INSTANT_COL if key == 'instant' else MEM_COL[key]
    lab = "instant adaptation" if key == 'instant' else rf"$\varepsilon$ = {key}"
    if key == 'instant':
        ax[0].plot(d.peak - d.threshold_bracket_hi, d.cycles, 'o-', ms=3, color=col, label=lab)
        continue
    fr = fitD[fitD.host == str(key)].iloc[0]
    dist = d.peak - fr.pc
    ax[0].plot(dist, d.cycles, 'o-', ms=3, color=col, label=lab)
    ax[1].plot(dist, (d.cycles - fr.a) * key * TD, 'o', ms=3.5, color=col, label=lab)
xx = np.logspace(-4, -0.8, 50)
ref = fitD[fitD.host == '0.003'].iloc[0]
ax[1].plot(xx, ref.b * 0.003 * TD * xx ** -0.5, color=INK, lw=1, ls='--', label=r"slope $-1/2$")
ax[0].set(xscale='log', yscale='log', xlabel="peak plaque above threshold",
          ylabel="cycles before conversion", title="Latency diverges at threshold")
ax[1].set(xscale='log', yscale='log', xlabel="peak plaque above threshold",
          ylabel=r"(cycles $-$ a) $\times\ \varepsilon T$", title="Collapse onto memory time")
ax[0].legend(fontsize=7); ax[1].legend(fontsize=7)
save_fig(fig, "fig07_04_latency_scaling")

## E. Debridement variant

At every reset, $M$ (the subgingival community) is set to 0 together with $u$, which stands for professional debridement. Neither intervention acts on $\theta$. Debridement always produces transient resolution, so entrapment cannot occur and the relevant quantity becomes the **relapse-free fraction of each maintenance interval**: the time after debridement at which $H$ rises above 0.5, divided by $T$ (1 if it never does).

In [ ]:
TE = 60.0; PEAKS_E = [0.60, 0.75]; NCE = 80
rows = []
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True, sharey=True)
for a_i, pk in enumerate(PEAKS_E):
    for key in ['instant', 0.01, 0.003]:
        mode, eps = ('instant', 0.0) if key == 'instant' else ('adaptive', key)
        s = simulate(peak_to_umax(pk, TE), TE, eps, n_cycles=NCE, mode=mode, reset_M=True)
        rt, fh = s['relapse_time'][0, :NCE], s['frac_high'][0, :NCE]
        rf = np.where(np.isnan(rt), np.where(fh > 0, 0.0, 1.0), rt / TE)
        for c in range(NCE):
            rows.append(dict(host=str(key), peak=pk, T=TE, cycle=c, relapse_free_fraction=rf[c],
                             theta_at_reset=s['theta_reset'][0, c], frac_cycle_inflamed=fh[c]))
        col = INSTANT_COL if key == 'instant' else MEM_COL[key]
        ax[a_i].plot(np.arange(NCE), rf, color=col,
                     label="instant adaptation" if key == 'instant' else rf"$\varepsilon$ = {key}")
    ax[a_i].set(xlabel="maintenance cycle (debridement at each reset)",
                title=f"peak plaque {pk} ({pk / U_UP:.2f} x static fold), T = 60")
ax[0].set_ylabel("relapse-free fraction of interval"); ax[0].legend(fontsize=7)
dfE = pd.DataFrame(rows); dfE.to_csv(EXP_DIR / "fig07_05_debridement_relapse_data.csv", index=False)
save_fig(fig, "fig07_05_debridement_relapse")
display(dfE[dfE.cycle.isin([0, 1, 2, 3, 5, 10, 20, 40, 79])].pivot_table(
    index='cycle', columns=['peak', 'host'], values='relapse_free_fraction').round(3))

## F. Consistency checks and summary

By the comparison principle for cooperative systems, conversion must be monotone in plaque at fixed $T$. Once a site converts at some peak, it converts at every higher peak, and no later. The checks below test this on every computed map.

In [ ]:
rows = []
for key, conv in maps.items():
    for iT, T in enumerate(T_LIST):
        c = conv[iT]; has = ~np.isnan(c)
        upward_closed = bool(np.all(has[np.argmax(has):])) if has.any() else True
        vals = c[has]
        nonincreasing = bool(np.all(np.diff(vals) <= 0)) if vals.size > 1 else True
        rows.append(dict(host='instant' if key == 'instant' else f'eps={key}', T=T,
                         conversion_set_upward_closed=upward_closed,
                         latency_nonincreasing_in_plaque=nonincreasing))
chk = pd.DataFrame(rows); chk.to_csv(EXP_DIR / "monotonicity_checks.csv", index=False)
print("all upward closed:", chk.conversion_set_upward_closed.all(),
      "| all latencies non-increasing:", chk.latency_nonincreasing_in_plaque.all())

iT60 = list(T_LIST).index(60.0)
summary = pd.DataFrame([
    dict(item="threshold at T=60, static fold (peak)", value=U_UP),
    dict(item="threshold at T=60, instant host (grid)", value=thr['instant'][iT60]),
    dict(item="threshold at T=60, eps->0 theory", value=peak_avg[iT60]),
    *[dict(item=f"threshold at T=60, eps={e} (grid)", value=thr[e][iT60]) for e in EPS_LIST],
    *[dict(item=f"latency fit exponent, eps={r.host}", value=r.exponent) for r in fitD.itertuples() if r.host != 'instant'],
    *[dict(item=f"latency amplitude b*eps*T, eps={r.host}", value=r.b_times_epsT) for r in fitD.itertuples() if r.host != 'instant'],
    dict(item="max |theta| diff RK4 vs LSODA", value=val_df.max_abs_theta_diff.max()),
])
summary.to_csv(EXP_DIR / "summary.csv", index=False)
display(summary)

In [ ]:
readme = """Experiment 07 - Reset ratchet
=============================
Model M1 (memory coordinates A = ln(theta0/theta)) under periodic plaque regrowth
u = umax(1 - exp(-k tau)), tau = t mod T, comparator backbone of Exp 06B.
Hosts: slow memory (eps = 0.3 ... 0.003); instant adaptation (A = alpha H, same equilibria);
frozen memory (eps = 0) for the adiabatic reduction. Naive initial site (0, 0, 0).
Entrapped cycle: H never below 0.5 within it. Conversion: first cycle from which all
later cycles are entrapped. Plaque is reported as the peak reached before each reset.

Files
- parameters.json, software_versions.txt
- validation_vs_LSODA.csv                   RK4 vs LSODA on four cases
- fig07_01_example_ratchets(.jpg/_data.csv) theta at reset and peak H per cycle, T=60
- frozen_memory_cycle_response.npz          Theta(A) on (T, peak, A) grid; low-branch mask; A*, A_c
- averaged_map_thresholds.csv               eps->0 thresholds from A* = alpha Theta(A*)
- fig07_02_reduced_map_and_thresholds       reduced map (T=60) and thresholds vs kT
- fig07_03_phase_maps                       cycles to conversion over (kT, peak) per host
- conversion_thresholds_peak_plaque.csv     grid thresholds per host and T
- fig07_04_latency_scaling, latency_scaling_fits.csv
- fig07_05_debridement_relapse              relapse-free fraction per cycle with M reset
- monotonicity_checks.csv, summary.csv
"""
with open(EXP_DIR / "README.txt", "w") as f:
    f.write(readme)
archive = shutil.make_archive(str(BASE_DIR / "Experiment_07_Reset_Ratchet_EXPORT"), 'zip', EXP_DIR)
print("Saved:", archive)